# CodeAgent-MCP Quick Start

在 Colab 中运行 CodeAgent-MCP 系统的快速入门。

In [ ]:
# Cell 1: 环境初始化
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/CodeAgent-MCP"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
# Cell 2: 安装依赖
!pip install -q openai mcp pydantic pyyaml rich faiss-cpu sentence-transformers gitpython
!pip install -q nest_asyncio pytest pytest-asyncio

import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Cell 3: 配置 API Key
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com"

# 验证
from openai import OpenAI
client = OpenAI()
resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "hello"}],
    max_tokens=5
)
print(f"API OK: {resp.choices[0].message.content}")

In [ ]:
# Cell 4: 同步代码 (首次用 git clone, 之后用 git pull)
# !git clone https://github.com/XIECHENG6/CodeAgent-MCP.git .
# !git pull origin main

In [ ]:
# Cell 5: 运行端到端测试 (不使用MCP工具)
import asyncio
from src.main import run

result = await run(
    "实现一个支持 get/put 操作的 LRU Cache, 要求 O(1) 时间复杂度",
    provider="default",
    use_mcp=False
)

In [ ]:
# Cell 6: 查看执行日志
import json
for entry in result.execution_log:
    stage = entry['stage']
    if stage == 'plan':
        print(f"=== PLAN ({len(entry['parsed_tasks'])} tasks) ===")
        for t in entry['parsed_tasks']:
            print(f"  {t['task_id']}: {t['description']}")
    elif stage == 'reviewer':
        r = entry['review']
        print(f"\n=== REVIEW (task {entry['task_id']}, attempt {entry['attempt']+1}) ===")
        print(f"  Score: {r['score']}/10, Passed: {r['passed']}")

In [ ]:
# Cell 7: 运行单元测试
!python -m pytest tests/test_orchestrator.py -v